In [1]:
import heapq
import copy

class PuzzleNode:
    def __init__(self, state, g=0, h=0, parent=None):
        self.state = state  # 3x3 tuple of tuples
        self.g = g          # Path cost (depth)
        self.h = h          # Heuristic cost
        self.f = g + h      # Total cost
        self.parent = parent

    def __lt__(self, other):
        return self.f < other.f

    def get_blank_pos(self):
        for r in range(3):
            for c in range(3):
                if self.state[r][c] == 0:
                    return r, c

    def generate_children(self):
        children = []
        r, c = self.get_blank_pos()
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                # Create new state
                new_state = [list(row) for row in self.state]
                # Swap blank (0) with the target tile
                new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]
                # Convert back to tuple of tuples (hashable)
                tuple_state = tuple(tuple(row) for row in new_state)
                children.append(PuzzleNode(tuple_state, g=self.g + 1, parent=self))
        return children

# Heuristic 1: Misplaced Tiles
def heuristic_misplaced(state, goal):
    misplaced = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0 and val != goal[r][c]:
                misplaced += 1
    return misplaced

# Heuristic 2: Manhattan Distance
def heuristic_manhattan(state, goal):
    # Precompute goal positions
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                gr, gc = goal_pos[val]
                distance += abs(r - gr) + abs(c - gc)
    return distance

def a_star(start_state, goal_state, heuristic_fn):
    start_node = PuzzleNode(start_state, g=0, h=heuristic_fn(start_state, goal_state))
    open_set = []
    heapq.heappush(open_set, start_node)
    closed_set = set()

    nodes_expanded = 0

    while open_set:
        current = heapq.heappop(open_set)
        nodes_expanded += 1

        if current.state == goal_state:
            # Reconstruct path
            path = []
            curr = current
            while curr:
                path.append(curr.state)
                curr = curr.parent
            return path[::-1], nodes_expanded

        closed_set.add(current.state)

        for child in current.generate_children():
            if child.state in closed_set:
                continue

            child.h = heuristic_fn(child.state, goal_state)
            child.f = child.g + child.h
            heapq.heappush(open_set, child)

    return None, nodes_expanded

# Define states (0 represents the empty space)
# Target configuration
goal = (
    (1, 2, 3),
    (8, 0, 4),
    (7, 6, 5)
)

# Starting configuration
start = (
    (2, 8, 3),
    (1, 6, 4),
    (7, 0, 5)
)

print("Solving using A* with Misplaced Tiles Heuristic...")
path_misplaced, nodes_misplaced = a_star(start, goal, heuristic_misplaced)
print(f"Steps to solve: {len(path_misplaced) - 1}")
print(f"Nodes expanded: {nodes_misplaced}")

print("\nSolving using A* with Manhattan Distance Heuristic...")
path_manhattan, nodes_manhattan = a_star(start, goal, heuristic_manhattan)
print(f"Steps to solve: {len(path_manhattan) - 1}")
print(f"Nodes expanded: {nodes_manhattan}")







# Print the step-by-step solution path for Manhattan Distance
print("Step-by-step solution path (Manhattan Distance):\n")
for i, step in enumerate(path_manhattan):
    print(f"Step {i}:")
    for row in step:
        print(list(row))
    print()

Solving using A* with Misplaced Tiles Heuristic...
Steps to solve: 5
Nodes expanded: 7

Solving using A* with Manhattan Distance Heuristic...
Steps to solve: 5
Nodes expanded: 6
Step-by-step solution path (Manhattan Distance):

Step 0:
[2, 8, 3]
[1, 6, 4]
[7, 0, 5]

Step 1:
[2, 8, 3]
[1, 0, 4]
[7, 6, 5]

Step 2:
[2, 0, 3]
[1, 8, 4]
[7, 6, 5]

Step 3:
[0, 2, 3]
[1, 8, 4]
[7, 6, 5]

Step 4:
[1, 2, 3]
[0, 8, 4]
[7, 6, 5]

Step 5:
[1, 2, 3]
[8, 0, 4]
[7, 6, 5]

